# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. 

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Convert metadata to a dict for pretty printing
metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}: {metadata_dict['description']}")
print(f"Version: {metadata_dict['version']}")


## 2. Data Overview

Review available record sets and all field/column `@id`s for reference.

Each RecordSet, Field, and Column is referred to **by its `@id`** for all subsequent data handling as per FAIR (and Croissant) best practice.

In [ ]:
print("Record Sets available in the dataset:")
for rec in dataset.record_sets:
    print(f"  - RecordSet @id: {rec.id}")
    print(f"    name: {rec.name}")
    print(f"    Fields:")
    for field in rec.fields:
        print(f"      - Field @id: {field.id}")
        print(f"        name: {field.name}")
        print(f"        dataType: {field.data_type}")
        if field.source_column is not None:
            print(f"        Column @id: {field.source_column.id}")
            print(f"          column name: {field.source_column.name}")
    print()

## 3. Data Extraction

Let us extract all record sets into pandas DataFrames for analysis. See above for available RecordSet, Field, and Column `@id` values.

Use the `@id` of each record set as keys in the dictionary.

In [ ]:
# List all record set @ids for processing
record_set_ids = [r.id for r in dataset.record_sets]
print("Record Sets to extract:", record_set_ids)

dataframes = dict()

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for {record_set_id}")
    print("Columns:", df.columns.tolist(), "\n")

# For demonstration, pick the first record set for EDA
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Preview of {main_rs_id} DataFrame:")
    display(dataframes[main_rs_id].head())
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)

Demonstration: select a numeric field (e.g., a questionnaire score), filter based on a threshold, normalize, and group by a categorical field.

All field/column access below is done via `@id` (as column names).

In [ ]:
if main_rs_id is not None and not dataframes[main_rs_id].empty:
    df = dataframes[main_rs_id]
    print("DataFrame columns:", df.columns.tolist())

    # List possible numeric fields (by field type)
    numeric_fields = []
    main_record_set = next((r for r in dataset.record_sets if r.id == main_rs_id), None)
    for field in main_record_set.fields:
        if field.data_type in ('Float', 'Number', 'Integer'):
            numeric_fields.append(field.id)
    print('Numeric fields (@id):', numeric_fields)

    # If GAD-7 or PHQ-9 total score present, use that; else pick first numeric
    preferred_score_fields = [f for f in numeric_fields if 'gad7' in f.lower() or 'phq9' in f.lower() or 'psq' in f.lower() or 'score' in f.lower()]
    if preferred_score_fields:
        numeric_field_id = preferred_score_fields[0]
    elif numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        print("No numeric fields found for EDA.")
        numeric_field_id = None

    # Pick a group field for demonstration
    candidate_groups = [f for f in main_record_set.fields if f.data_type in ('Text', 'String') and (('sex' in f.id.lower()) or ("gender" in f.id.lower()) or ("age" in f.id.lower()) or ("village" in f.id.lower()) or ("education" in f.id.lower()))]
    if candidate_groups:
        group_field_id = candidate_groups[0].id
    else:
        # Fallback to first text/string field
        text_fields = [f.id for f in main_record_set.fields if f.data_type in ('Text', 'String')]
        if text_fields:
            group_field_id = text_fields[0]
        else:
            group_field_id = None

    if numeric_field_id is not None and numeric_field_id in df.columns:
        # Convert to numeric (handle string data)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors="coerce")

        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records in '{main_rs_id}' where {numeric_field_id} > {threshold}:")
        display(filtered_df[[numeric_field_id] + ([group_field_id] if group_field_id is not None and group_field_id in filtered_df.columns else [])].head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric field found or present in DataFrame columns.")
else:
    print("No record set DataFrame available for EDA.")

## 5. Visualization

Let's visualize the distribution of a questionnaire score (if present), and the group-wise average (e.g. by sex/gender or village).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if main_rs_id is not None and not dataframes[main_rs_id].empty and numeric_field_id is not None and numeric_field_id in dataframes[main_rs_id]:
    df = dataframes[main_rs_id]
    # Plot distribution of the score
    plt.figure(figsize=(8,4))
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We have loaded the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the Croissant schema and `mlcroissant`. 

- All dataset components (RecordSets, Fields, Columns) were referenced and handled by their `@id`.
- Data were extracted into pandas DataFrames for flexible analysis.
- We previewed records, performed basic filtering, normalization, and group-wise aggregation.
- Simple visualizations illustrated value distributions and group trends.

This workflow can be continued to explore deeper patterns and relationships in the data for research and public health applications.
